# Eye EPA 라벨 조합 실험
A: 1·2=1, 3·4=0. B: 1·2·3=1, 4=0. comp_group_id와 split_group_id를 공유하는 행을 연결 그룹으로 묶습니다. 공통 구조 특성 2265개, 그룹 외부5/내부3 교차검증, CatBoost를 제외한 6개 모델·각 20개 설정. 전처리·가중치·임계값은 학습 fold 안에서만 결정합니다.

In [1]:
import os, sys
from pathlib import Path
OUTPUT_ROOT = Path(r'C:/Users/DS/COS/최종/label 조합 변경')
sys.path.insert(0, str(OUTPUT_ROOT))
from epa_experiment_utils import run_endpoint, verify_endpoint
RUN_ID = os.environ.setdefault('EPA_RUN_ID', 'epa-production-20261106-no-catboost')
CSV_PATH = Path(r'C:/Users/DS/COS/최종/production데이터/eye_vectorized.csv')
{'endpoint':'eye','run_id':RUN_ID,'input':str(CSV_PATH),'output':str(OUTPUT_ROOT)}

{'endpoint': 'eye',
 'run_id': 'epa-production-20261106-no-catboost',
 'input': 'C:\\Users\\DS\\COS\\최종\\production데이터\\eye_vectorized.csv',
 'output': 'C:\\Users\\DS\\COS\\최종\\label 조합 변경'}

In [2]:
result = run_endpoint('eye', CSV_PATH, output_dir=OUTPUT_ROOT, run_id=RUN_ID)
verify_endpoint(result['output_dir'])

eye: 673 rows, 2265 features; artifacts=C:\Users\DS\COS\최종\label 조합 변경\runs\epa-production-20261106-no-catboost\eye\bb1eca6656e3324e


DONE eye/A/lr outer=1/5 elapsed=1.3s


DONE eye/A/lr outer=2/5 elapsed=1.3s


DONE eye/A/lr outer=3/5 elapsed=1.3s


DONE eye/A/lr outer=4/5 elapsed=1.3s


DONE eye/A/lr outer=5/5 elapsed=1.3s


... (52 outputs left out)

DONE eye/B/lgbm outer=3/5 elapsed=1.7s

DONE eye/B/lgbm outer=4/5 elapsed=1.7s


DONE eye/B/lgbm outer=5/5 elapsed=1.7s


{'endpoint': 'eye',
 'identity': 'bb1eca6656e3324ef9d7303f3c121fbff2036d93c499ec5b20b5fb6a799e3cb6',
 'combinations': 12,
 'outer_folds': 60,
 'status': 'passed'}

In [3]:
import pandas as pd
columns = ['endpoint', 'label', 'model', 'sensitivity', 'specificity', 'MCC', 'ROC-AUC',
           'abs_gap', 'PR-AUC', 'precision', 'F1', 'balanced_accuracy',
           'positive_count', 'negative_count', 'confusion_matrix',
           'sensitivity_std', 'specificity_std', 'MCC_std', 'ROC-AUC_std']
report = pd.DataFrame(result['report'])
for metric in ('sensitivity', 'specificity', 'MCC', 'ROC-AUC'):
    report[metric + '_std'] = report['fold_dispersion'].map(lambda values: values[metric])
report[columns]

,endpoint,label,model,sensitivity,specificity,MCC,ROC-AUC,abs_gap,PR-AUC,precision,F1,balanced_accuracy,positive_count,negative_count,confusion_matrix,sensitivity_std,specificity_std,MCC_std,ROC-AUC_std
0,eye,A,lr,0.728000,0.737226,0.379523,0.738350,0.009226,0.389376,0.387234,0.505556,0.732613,125,548,"[[404, 144], [34, 91]]",0.110995,0.115325,0.114102,0.062593
1,eye,A,rf,0.672000,0.680657,0.281869,0.717956,0.008657,0.378600,0.324324,0.437500,0.676328,125,548,"[[373, 175], [41, 84]]",0.103537,0.122291,0.085631,0.070531
2,eye,A,et,0.648000,0.680657,0.263269,0.692161,0.032657,0.341175,0.316406,0.425197,0.664328,125,548,"[[373, 175], [44, 81]]",0.107331,0.068122,0.117560,0.095948
3,eye,A,svm,0.448000,0.828467,0.258340,0.669912,0.380467,0.331992,0.373333,0.407273,0.638234,125,548,"[[454, 94], [69, 56]]",0.414150,0.183952,0.245103,0.091091
4,eye,A,xgb,0.728000,0.614964,0.268165,0.677182,0.113036,0.375898,0.301325,0.426230,0.671482,125,548,"[[337, 211], [34, 91]]",0.130843,0.175693,0.141707,0.089151
5,eye,A,lgbm,0.728000,0.609489,0.263652,0.693672,0.118511,0.369978,0.298361,0.423256,0.668745,125,548,"[[334, 214], [34, 91]]",0.181989,0.099973,0.194589,0.112034
6,eye,B,lr,0.635294,0.576613,0.205798,0.611252,0.058681,0.682056,0.720000,0.675000,0.605954,425,248,"[[143, 105], [155, 270]]",0.022010,0.089612,0.104406,0.070929
7,eye,B,rf,0.654118,0.657258,0.301350,0.656044,0.003140,0.741707,0.765840,0.705584,0.655688,425,248,"[[163, 85], [147, 278]]",0.059175,0.097808,0.046140,0.040272
8,eye,B,et,0.656471,0.616935,0.265434,0.645332,0.039535,0.725971,0.745989,0.698373,0.636703,425,248,"[[153, 95], [146, 279]]",0.064115,0.066105,0.066656,0.037333
9,eye,B,svm,0.609412,0.556452,0.160776,0.596319,0.052960,0.685321,0.701897,0.652393,0.582932,425,248,"[[138, 110], [166, 259]]",0.216195,0.190723,0.059394,0.033553


## 해석
PR-AUC는 Average Precision(AP)이며, 각 `_std` 열은 외부 5-fold의 표본 표준편차(ddof=1)입니다.
SVM의 fold별 margin은 공통 확률 척도로 보정되지 않았으므로 pooled ROC-AUC·PR-AUC는 기술적 순위 요약으로만 해석합니다.
sensitivity·specificity·MCC·ROC-AUC를 함께 확인합니다. sensitivity와 specificity의 절대 수준, 차이, fold 변동성으로 균형을 판단합니다. 단일 지표 최고 모델을 선언하지 않으며 gap만 낮은 결과도 좋은 모델로 해석하지 않습니다. A/B는 다른 예측 문제입니다. SVM fold 간 margin은 보정된 확률이 아니므로 통합 ROC/PR은 주의해서 해석합니다.